# Part 1: Web Scraping Benchmark — BeautifulSoup vs Playwright

**Course:** CSIS 4260 — Douglas College  
**Student:** Desmond Chua  

This notebook compares two popular Python web scraping libraries — **BeautifulSoup** and **Playwright** —
by scraping the [r/healthIT](https://www.reddit.com/r/healthIT/) subreddit, a community of healthcare IT professionals.

We will fetch 10 posts from Reddit's public JSON API and benchmark each library on speed, ease of use, and suitability for different scraping tasks.

## Library 1: BeautifulSoup

BeautifulSoup (bs4) is a Python library for pulling data out of HTML and XML documents. It works with a parser to provide idiomatic ways of navigating, searching, and modifying the parse tree.

- **Lightweight and fast** — it doesn't launch a browser, so it has very low overhead and is ideal for simple HTTP requests
- **Easy to learn** — the API is beginner-friendly and well-documented, making it a great first scraping library
- **Limited to static content** — it can only parse what the server returns in the initial HTML/JSON response; it cannot execute JavaScript
- **Best for structured APIs and static pages** — perfect when the data is already available in the page source or via a JSON endpoint (like Reddit's `.json` API)

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import json

# The Reddit JSON API endpoint — adding .json to any subreddit URL returns structured data
url = "https://www.reddit.com/r/healthIT/.json?limit=10"

# Reddit requires a descriptive User-Agent header or it will block our request
headers = {"User-Agent": "healthcare-nlp-assignment/1.0 (by /u/Similar_Road_6567)"}

# Start the timer so we can measure how long the scraping takes
start_time = time.time()

# Send the HTTP GET request to Reddit's JSON API
response = requests.get(url, headers=headers)

# Parse the raw response text as HTML — even though it's JSON, BS4 can still parse it
soup = BeautifulSoup(response.text, "html.parser")

# Extract the actual JSON data from the parsed page text
raw_json = json.loads(soup.get_text())

# Reddit wraps posts inside data -> children -> each child has a "data" dict
posts = raw_json["data"]["children"]

# Store our scraped results in a simple list
bs4_results = []

for post in posts:
    post_data = post["data"]

    title = post_data["title"]
    post_url = post_data["url"]

    # Some posts are link-only and have no selftext — use empty string as fallback
    if post_data["selftext"]:
        selftext = post_data["selftext"]
    else:
        selftext = ""

    bs4_results.append({"title": title, "url": post_url, "selftext": selftext})

# Stop the timer
end_time = time.time()

# Calculate how long the entire scraping process took
bs4_time = end_time - start_time

# Print each post title so we can verify the scrape worked
for result in bs4_results:
    print(result["title"])

print(f"\nBeautifulSoup total time: {bs4_time:.2f} seconds")
print(f"Posts scraped: {len(bs4_results)}")

c:\Users\user\OneDrive\Desmond_New\healthcare-nlp-analysis\venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


"I want to be an Epic analyst" FAQ
Anyone here switch careers without a degree in informatics?
I built a browser-based ambient scribe that keeps all data on the device (open source)
How to generate a list of patients in EPIC based on ICD codes for research?
Epic Clarity Exam-Granularity
client wanted a healthcare app "like uber but for doctors". here's how that went
Records sent to another doctor via MyChart?
HCA Medical Billing
Info about HL7FHIR reference format
Epic Ambulatory Recertification

BeautifulSoup total time: 2.09 seconds
Posts scraped: 10


## Library 2: Playwright

Playwright is a browser automation library developed by Microsoft. It launches a real browser (Chromium, Firefox, or WebKit) and can interact with web pages just like a human user would.

- **Handles JavaScript-rendered content** — because it runs a real browser, it can scrape pages that load data dynamically via JavaScript (SPAs, infinite scroll, etc.)
- **Slower and heavier** — launching a full browser adds significant overhead compared to a simple HTTP request, which makes it overkill for static content or JSON APIs
- **Powerful automation features** — it can click buttons, fill forms, take screenshots, and wait for elements to load, making it great for complex scraping workflows
- **Best for modern web apps** — ideal when the data you need is only available after JavaScript executes, such as dashboards, single-page applications, or sites with anti-bot protections

In [6]:
import os
print(os.getcwd())

# The notebook already runs from inside the notebooks folder
result = subprocess.run(
    ["python", "playwright_benchmark.py"],
    capture_output=True,
    text=True,
    cwd=os.getcwd()
)

print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

c:\Users\user\OneDrive\Desmond_New\healthcare-nlp-analysis\notebooks
"I want to be an Epic analyst" FAQ
Anyone here switch careers without a degree in informatics?
I built a browser-based ambient scribe that keeps all data on the device (open source)
How to generate a list of patients in EPIC based on ICD codes for research?
Epic Clarity Exam-Granularity
client wanted a healthcare app "like uber but for doctors". here's how that went
Records sent to another doctor via MyChart?
HCA Medical Billing
Info about HL7FHIR reference format
Epic Ambulatory Recertification

Playwright total time: 16.07 seconds
Posts scraped: 10



## Benchmark Results

In [8]:
from tabulate import tabulate

# Hardcoding the benchmark times we recorded from running Cells 3 and 5
bs4_time = 2.47
playwright_time = 16.07

# Build a comparison table with key metrics for each library
results = {
    "Library": ["BeautifulSoup", "Playwright"],
    "Time (sec)": [round(bs4_time, 2), round(playwright_time, 2)],
    "Lines of Code": [15, 22],
    "Ease of Use (1-5)": [5, 3],
    "Best For": ["Static pages & JSON APIs", "JavaScript-rendered pages"]
}

# Convert the dict into a list of rows for tabulate
table_rows = []
for i in range(len(results["Library"])):
    row = [results[col][i] for col in results]
    table_rows.append(row)

# Print a nicely formatted comparison table
print(tabulate(table_rows, headers=results.keys(), tablefmt="grid"))

# Print our recommendation based on the benchmark results
print("\n--- Recommendation ---")
print("For this project, BeautifulSoup is the better choice. It is significantly faster")
print("and simpler for scraping Reddit's JSON API, which returns structured data that")
print("does not require JavaScript rendering. Playwright only adds value when the target")
print("website relies on JavaScript to load its content dynamically.")

+---------------+--------------+-----------------+---------------------+---------------------------+
| Library       |   Time (sec) |   Lines of Code |   Ease of Use (1-5) | Best For                  |
+===============+==============+=================+=====================+===========================+
| BeautifulSoup |         2.47 |              15 |                   5 | Static pages & JSON APIs  |
+---------------+--------------+-----------------+---------------------+---------------------------+
| Playwright    |        16.07 |              22 |                   3 | JavaScript-rendered pages |
+---------------+--------------+-----------------+---------------------+---------------------------+

--- Recommendation ---
For this project, BeautifulSoup is the better choice. It is significantly faster
and simpler for scraping Reddit's JSON API, which returns structured data that
does not require JavaScript rendering. Playwright only adds value when the target
website relies on JavaScrip